In [ ]:
from pathlib import Path
import random, math, re, time
import numpy as np
from tqdm import tqdm
from fabio.cbfimage import CbfImage
import fabio
import multiprocessing

In [ ]:
random.seed(979969114)
np.random.seed(979969114)

In [ ]:
_SIZE_RE = re.compile(r'(?i)^(\d+(?:\.\d+)?)([kmgt]?)b?$')
def parse_size(txt: str) -> int:
    m = _SIZE_RE.match(txt.strip())
    if not m:
        raise ValueError(f'Bad size spec: {txt}')
    val, unit = m.groups()
    return int(float(val) * {'':1,'k':2**10,'m':2**20,'g':2**30,'t':2**40}[unit.lower()])

def load_seed_arrays(seed_path: Path, bitdepth: int):
    seeds = []
    for p in sorted(seed_path.glob("*.cbf")):
        arr = fabio.open(str(p)).data.astype(
            np.uint16 if bitdepth == 16 else np.uint32
        )
        seeds.append(arr)
    if not seeds:
        raise FileNotFoundError(f"No *.cbf images found under {seed_path}")
    return seeds

def jitter(img: np.ndarray) -> np.ndarray:
    out = img.astype(np.float32)
    out *= random.uniform(0.85, 1.15)
    mean = out.mean()
    out  = (out - mean) * random.uniform(0.9, 1.1) + mean
    if random.random() < 0.8:
        shifts = (random.randint(-16,16), random.randint(-16,16))
        out = np.roll(out, shift=shifts, axis=(0,1))
    if random.random() < 0.9:
        out += np.random.normal(0, 20, size=out.shape)
    maxv = 65535 if img.dtype == np.uint16 else 2**32-1
    np.clip(out, 0, maxv, out=out)
    return out.astype(img.dtype)

def write_cbf(arr: np.ndarray, path: Path):
    img = fabio.cbfimage.CbfImage(data=arr.astype(np.uint16))
    img.write(str(path))

def _init_worker(seed_dir: str, bitdepth: int):
    global SEEDS, BITDEPTH
    BITDEPTH = bitdepth
    SEEDS = load_seed_arrays(Path(seed_dir), bitdepth)

def _worker(idx_tuple):
    idx, out_dir = idx_tuple
    img = jitter(random.choice(SEEDS))
    dst = Path(out_dir) / f"slice_{idx:07d}.cbf"
    write_cbf(img, dst)
    return dst.stat().st_size

def generate_parallel(
    seed_dir,
    out_dir,
    target_size: str = "1T",
    bitdepth: int = 16,
    workers: int = None
):
    seed_dir = Path(seed_dir)
    out_dir  = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    seeds = load_seed_arrays(seed_dir, bitdepth)
    raw_bytes = seeds[0].nbytes
    total_bytes = parse_size(target_size)

    existing_files = list(out_dir.glob("slice_*.cbf"))
    existing_count = len(existing_files)
    existing_bytes = sum(p.stat().st_size for p in existing_files)

    if existing_count > 0:
        avg_bytes = existing_bytes / existing_count
    else:
        sample_arr  = jitter(seeds[0])
        sample_path = out_dir / "._cbf_sample.cbf"
        write_cbf(sample_arr, sample_path)
        avg_bytes = sample_path.stat().st_size
        sample_path.unlink()

    if existing_bytes >= total_bytes:
        print(f"✅ Already {existing_count} slices "
              f"({existing_bytes/2**30:.1f} GB) ≥ target {target_size}")
        return

    remaining = total_bytes - existing_bytes
    n_remaining = math.ceil(remaining / avg_bytes)
    start_idx   = (max(int(p.stem.split("_")[1]) for p in existing_files) + 1) \
                  if existing_count else 0

    print(f"⏳ Have {existing_count} slices ({existing_bytes/2**30:.1f} GB); "
          f"avg slice = {avg_bytes/2**20:.1f} MB; "
          f"generating {n_remaining} more → ~{total_bytes/2**30:.1f} GB")

    tasks = [(start_idx + i, out_dir) for i in range(n_remaining)]
    n_workers = multiprocessing.cpu_count()

    t0 = time.time()
    produced = 0
    with multiprocessing.Pool(
        processes=n_workers,
        initializer=_init_worker,
        initargs=(str(seed_dir), bitdepth)
    ) as pool:
        for fsize in tqdm(
            pool.imap_unordered(_worker, tasks),
            total=n_remaining,
            unit="slice"
        ):
            produced += 1
        pool.close()
        pool.join()

    elapsed = time.time() - t0
    final_count = existing_count + produced
    final_bytes = existing_bytes + produced * avg_bytes
    gbps = (produced * avg_bytes / 1e9) / elapsed

    print(f"✅ Done in {elapsed/60:.1f} min — added {produced} slices "
          f"({produced*avg_bytes/2**30:.1f} GB), avg {gbps:.2f} GB/s")
    print(f"   → Total: {final_count} slices ({final_bytes/2**30:.1f} GB)")